# Marketing Goals — **CV → OOS predictive backtest**

*Lee Jerusalmy*

Isolated experiment: `experiments/cv_optimization/cv_oos_backtest/`.

**Question:** Does high in-sample CV mean the Marketing Goal is unreliable out-of-sample?

Walk-forward / rolling historical backtest of the **existing** methodology (no look-ahead).  
Does **not** change CV, trim, windows, thresholds, or goal formulas.

**Patch goal definition (documented):** `mean_after` from `patch_cv_adaptive`  
= weighted mean of kept cohort `growth_ratio = ARPU_e/ARPU_s` — the quantity existing CV describes.

Other experiments (`robust_cv`, `cv_diagnosis`, …) are untouched.


In [1]:
# ── Auth (Google Colab) ──────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
IN_COLAB = True
print('Authenticated (Colab)')


Authenticated (Colab)


In [2]:
project_id = 'oceanic-citadel-454608-d2'
from google.cloud import bigquery
client = bigquery.Client(project=project_id)
print('BigQuery client ready for', project_id)


BigQuery client ready for oceanic-citadel-454608-d2


In [3]:
# ── Optional: mount Google Drive (to write under marketing_goals/runs/) ──
# Skip this cell if you only want browser downloads.

MOUNT_DRIVE = True  # set False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted. Expected runs path:')
    print('  /content/drive/MyDrive/lee_project/marketing_goals/runs')
    print('  (or My Drive with a space — export cell checks both)')
else:
    print('Drive mount skipped — export will browser-download CSVs only.')


Mounted at /content/drive
Drive mounted. Expected runs path:
  /content/drive/MyDrive/lee_project/marketing_goals/runs
  (or My Drive with a space — export cell checks both)


In [4]:
import pandas as pd
import numpy as np
from pandas_gbq import read_gbq

# ── Date anchor (shared for both brands) ──
# CHANGED (temporary): evaluation "today" for OOS maturity ceiling + comparability
# REVERT for production rolling logic later.
AS_OF_DATE = pd.Timestamp('2026-08-03')

# ── Shared patch / horizon calendar ──
PATCHES = (
    (1, 7), (7, 14), (14, 30), (30, 60), (60, 90),
    (90, 120), (120, 150), (150, 180), (180, 270), (270, 365),
)
GOAL_HORIZONS = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
CHECKPOINTS   = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
LOOKBACK_COHORTS = 35  # unchanged production lookback
EXPERIMENT_TAG = 'cv_oos_backtest'

# ADDED: walk-forward knobs (framework only — not production model knobs)
TEST_HORIZONS = [7, 14, 30]          # next N cohort dates after training_end
PRIMARY_TEST_HORIZON = 14            # main analysis horizon
CUTOFF_FREQ_DAYS = 14                # biweekly historical cutoffs (raise runtime if weekly)
CUTOFF_SPAN_DAYS = 180               # how far back cutoffs extend
ORGANIC_LABEL = 'Organic'

# ── Which brands to run ──
RUN_BRANDS = ['realprize', 'lonestar']

# ── Per-brand knobs (from config/*.yaml / Combined predecessors) ──
BRAND_CONFIGS = {
    'realprize': {
        'brand': 'realprize',
        'cost_table': 'analytics.realprize_cost_per_user',
        'deposits_table': 'realprize.casino_astropay_dmn',
        # TikTok excluded entirely
        'exclude_affids': [4313],
        'populations': ['Web', 'App', 'Affiliate'],
        'trim_config': {
            'App':       {'method': 'winsor', 'pct': 0},
            'Web':       {'method': 'winsor', 'pct': 0.01},
            'Affiliate': {'method': 'winsor', 'pct': 0.01},
            'Blended':   {'method': 'winsor', 'pct': 0},
        },
        'organic_trim_method': 'winsor',
        'organic_trim_pct': 0,
        # App attribution change ~2025-08-12 — pin share above this horizon
        'organic_share_cap_horizon': 120,
        'cv_threshold': 0.15,
        'cv_good_enough': 0.10,
        'max_remove_fraction': 0.15,
        # RP Combined did not use a min-cohort-dates gate (default 1)
        'min_cohort_dates': 1,
        # Tail fill only in LS predecessor
        'extrapolate_tail': False,
        'extrapolation_tail_days': 30,
        'has_scope_bucket': True,  # app / non_app + app_organic
        'users_sql': None,  # filled below via builder
    },
    'lonestar': {
        'brand': 'lonestar',
        'cost_table': 'analytics.lonestar_cost_per_user',
        'deposits_table': 'lonestar.casino_astropay_dmn',
        # TikTok + TikTok Canada
        'exclude_affids': [4866, 7127],
        'populations': ['Web', 'Affiliate'],  # add App when LS App launches
        'trim_config': {
            'Web':       {'method': 'winsor', 'pct': 0},
            'Affiliate': {'method': 'winsor', 'pct': 0.01},
            'Blended':   {'method': 'winsor', 'pct': 0},
        },
        'organic_trim_method': 'winsor',
        'organic_trim_pct': 0,
        # No organic share cap on LS
        'organic_share_cap_horizon': None,
        'cv_threshold': 0.175,
        'cv_good_enough': 0.10,
        'max_remove_fraction': 0.15,
        'min_cohort_dates': 20,
        'extrapolate_tail': True,
        'extrapolation_tail_days': 30,
        'has_scope_bucket': False,  # scope defaults to 'all' in organic helper
        'users_sql': None,
    },
}

# ── Monitoring (print sub-outputs after each pipeline step) ──
# True  = show intermediate tables in Colab after each part (recommended first runs)
# False = quieter (still prints step banners + row counts)
MONITOR_STEPS = True
MONITOR_PREVIEW_DAYS = [1, 7, 14, 30, 60, 90, 120, 180, 270, 365]
MONITOR_PREVIEW_HORIZONS = [7, 30, 120, 365]  # goals sample only

print('ENV: Google Colab')

# ── Seed Combined globals (overwritten per brand in apply_brand_globals) ──
# Helper cells use these names as defaults; they MUST exist before those cells run.
POPULATIONS = list(BRAND_CONFIGS['realprize']['populations'])  # first brand's list as placeholder
TRIM_CONFIG = dict(BRAND_CONFIGS['realprize']['trim_config'])
ORGANIC_TRIM_METHOD = BRAND_CONFIGS['realprize']['organic_trim_method']
ORGANIC_TRIM_PCT = BRAND_CONFIGS['realprize']['organic_trim_pct']
ORGANIC_SHARE_CAP_HORIZON = BRAND_CONFIGS['realprize']['organic_share_cap_horizon']
CV_THRESHOLD = BRAND_CONFIGS['realprize']['cv_threshold']
CV_GOOD_ENOUGH = BRAND_CONFIGS['realprize']['cv_good_enough']
MAX_REMOVE_FRACTION = BRAND_CONFIGS['realprize']['max_remove_fraction']
MIN_COHORT_DATES = BRAND_CONFIGS['realprize']['min_cohort_dates']
EXTRAPOLATE_TAIL = BRAND_CONFIGS['realprize']['extrapolate_tail']
EXTRAPOLATION_TAIL_DAYS = BRAND_CONFIGS['realprize']['extrapolation_tail_days']
ACTIVE_BRAND = None

print(f'Config loaded. as_of_date = {AS_OF_DATE.date()}')
print(f'Brands to run: {RUN_BRANDS}')
print(f'Monitor steps = {MONITOR_STEPS}')
print('Mode: PERSISTENT TRIM (excluded users carry forward)')
print('Main goals columns: brand, population, goal_horizon, day, raw_goal_ratio, organic_share, adjusted_goal_ratio')


ENV: Google Colab
Config loaded. as_of_date = 2026-08-03
Brands to run: ['realprize', 'lonestar']
Monitor steps = True
Mode: PERSISTENT TRIM (excluded users carry forward)
Main goals columns: brand, population, goal_horizon, day, raw_goal_ratio, organic_share, adjusted_goal_ratio


In [5]:
# ══════════════════════════════════════════════════════════════
# APPLY BRAND CONFIG → module-level globals used by helpers
# (same variable names as Combined predecessors)
# ══════════════════════════════════════════════════════════════

def apply_brand_globals(cfg: dict) -> None:
    """Set Combined-style globals for one brand before running the pipeline."""
    global POPULATIONS, TRIM_CONFIG
    global ORGANIC_TRIM_METHOD, ORGANIC_TRIM_PCT, ORGANIC_SHARE_CAP_HORIZON
    global CV_THRESHOLD, CV_GOOD_ENOUGH, MAX_REMOVE_FRACTION
    global MIN_COHORT_DATES, EXTRAPOLATE_TAIL, EXTRAPOLATION_TAIL_DAYS
    global ACTIVE_BRAND

    ACTIVE_BRAND = cfg['brand']
    POPULATIONS = list(cfg['populations'])
    TRIM_CONFIG = dict(cfg['trim_config'])
    ORGANIC_TRIM_METHOD = cfg['organic_trim_method']
    ORGANIC_TRIM_PCT = cfg['organic_trim_pct']
    ORGANIC_SHARE_CAP_HORIZON = cfg['organic_share_cap_horizon']
    CV_THRESHOLD = cfg['cv_threshold']
    CV_GOOD_ENOUGH = cfg['cv_good_enough']
    MAX_REMOVE_FRACTION = cfg['max_remove_fraction']
    MIN_COHORT_DATES = cfg['min_cohort_dates']
    EXTRAPOLATE_TAIL = cfg['extrapolate_tail']
    EXTRAPOLATION_TAIL_DAYS = cfg['extrapolation_tail_days']

    print(f"\n{'=' * 60}")
    print(f"BRAND: {ACTIVE_BRAND}")
    print(f"  populations = {POPULATIONS}")
    print(f"  CV threshold = {CV_THRESHOLD}  good_enough = {CV_GOOD_ENOUGH}")
    print(f"  min_cohort_dates = {MIN_COHORT_DATES}")
    print(f"  extrapolate_tail = {EXTRAPOLATE_TAIL}")
    print(f"  organic_share_cap_horizon = {ORGANIC_SHARE_CAP_HORIZON}")
    print(f"{'=' * 60}")


def load_brand_tables(cfg: dict, as_of_date=AS_OF_DATE):
    """Pull cost-per-user population assignment + deposit revenue for one brand."""
    # ADDED (backtest framework): deeper SQL floor so early cutoffs have history.
    # Does not change patch lookback (still LOOKBACK_COHORTS=35 at each cutoff).
    _extra = int(globals().get('CUTOFF_SPAN_DAYS', 0)) + int(max(globals().get('TEST_HORIZONS', [30]))) + 40
    sql_floor = (
        as_of_date - pd.Timedelta(days=max(GOAL_HORIZONS) + LOOKBACK_COHORTS + _extra)
    ).date()
    brand = cfg['brand']
    cost_table = cfg['cost_table']
    dep_table = cfg['deposits_table']
    excl = cfg['exclude_affids']
    excl_sql = ', '.join(str(a) for a in excl)

    print(f'[{brand}] SQL floor date: {sql_floor}  (AS_OF_DATE = {as_of_date.date()})')

    if brand == 'realprize':
        # scope + bucket for App organic vs non_app organic
        users_sql = f"""
        SELECT
          id,
          CASE
            WHEN affid IN (63, 2521, 2535, 4957, 4971, 5048, 5062, 5069) THEN 'Web'
            WHEN affid = 1                                    THEN 'App'
            WHEN affid IN (64, 71)                            THEN 'PPC'
            WHEN affid IN (0, 78, 2290)                       THEN 'Organic'
            ELSE 'Affiliate'
          END AS population,
          CASE WHEN affid = 1 THEN 'app' ELSE 'non_app' END AS scope,
          CASE
            WHEN affid = 1 AND channel_type = 'app_organic' THEN 'organic'
            WHEN affid = 1                                   THEN 'acquired'
            WHEN affid IN (0, 78, 2290)                      THEN 'organic'
            ELSE 'acquired'
          END AS bucket,
          DATE(MIN(cost_date)) AS cost_date
        FROM `{cost_table}`
        WHERE cost_date >= DATE('{sql_floor}')
          AND affid NOT IN ({excl_sql})
          AND id > 0
        GROUP BY id, population, scope, bucket
        """
    elif brand == 'lonestar':
        # No scope/bucket — organic helper defaults to scope='all'
        users_sql = f"""
        SELECT
          id,
          CASE
            WHEN affid IN (63, 4432, 4551, 4698, 5048, 5125, 7120, 7253, 7260, 8331, 8345) THEN 'Web'
            -- WHEN affid = 1 THEN 'App'   -- uncomment when LS App launches
            WHEN affid IN (64, 71)  THEN 'PPC'
            WHEN affid IN (0, 78)   THEN 'Organic'
            ELSE 'Affiliate'
          END AS population,
          DATE(MIN(cost_date)) AS cost_date
        FROM `{cost_table}`
        WHERE cost_date >= DATE('{sql_floor}')
          AND affid NOT IN ({excl_sql})
          AND id > 0
        GROUP BY 1, 2
        """
    else:
        raise ValueError(f'Unknown brand: {brand}')

    revenue_sql = f"""
    SELECT
      playerId AS playerid,
      DATE(date) AS date,
      SUM(amount) / 100.0 AS amount
    FROM `{dep_table}`
    WHERE Status = 'APPROVED'
      AND date >= DATE('{sql_floor}')
    GROUP BY 1, 2
    """

    users_df = read_gbq(users_sql, project_id=project_id, use_bqstorage_api=True)
    revenue_df = read_gbq(revenue_sql, project_id=project_id, use_bqstorage_api=True)

    print(f'[{brand}] users_df:   {len(users_df):,} rows')
    print(f'[{brand}] revenue_df: {len(revenue_df):,} rows')
    print(users_df['population'].value_counts().to_string())
    return users_df, revenue_df

print('Brand loader + apply_brand_globals ready.')


Brand loader + apply_brand_globals ready.


In [6]:
# ══════════════════════════════════════════════════════════════
# HELPERS — math & base table construction
# ══════════════════════════════════════════════════════════════

def weighted_mean_std_cv(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    mu  = np.average(x, weights=w)
    var = np.average((x - mu) ** 2, weights=w)
    sd  = np.sqrt(var)
    cv  = sd / mu if mu != 0 else np.nan
    return mu, sd, cv


def build_user_revenue_cums(users_df, revenue_df, *, max_day=365):
    """
    Precompute cumulative per-user revenue indexed by (population, cost_date, user, dsi).
    Returns:
      u          — one row per (population, user) with their earliest cost_date
      daily_user — cumulative revenue at each (population, cost_date, user, dsi)
    """
    u = users_df[['id', 'population', 'cost_date']].copy()
    u['population'] = u['population'].astype(str).str.strip()
    u['cost_date']  = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()
    u = u.groupby(['population', 'id'], as_index=False)['cost_date'].min()
    u = u.rename(columns={'id': '__uid__'})

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()

    rr = r.merge(u, left_on='playerid', right_on='__uid__', how='inner')
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max_day - 1))].copy()

    daily_user = (
        rr.groupby(['population', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['population', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['population', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )
    return u, daily_user


print('Math + base table helpers defined.')


Math + base table helpers defined.


In [7]:
# ══════════════════════════════════════════════════════════════
# HELPERS — trimming & cohort revenue summation
# ══════════════════════════════════════════════════════════════

def compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=0.01):
    """Cap per-user cumulative revenue at the (1-top_pct) quantile within each cohort date."""
    if top_pct <= 0:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    per_user['cap_e'] = (
        per_user.groupby(['population', 'cost_date'], observed=True)['cum_e']
                .transform(lambda s: s[s > 0].quantile(1.0 - top_pct) if (s > 0).any() else np.inf)
    )
    return per_user[['population', 'cost_date', '__uid__', 'cap_e']]


def apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=0.10):
    """Remove the top trim_pct of depositors (by cumulative revenue) from each cohort date."""
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        return cohort_users.copy()
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    depositors = per_user.loc[per_user['cum_e'] > 0].copy()
    if depositors.empty:
        return cohort_users.copy()
    thresholds = (
        depositors.groupby(['population', 'cost_date'], observed=True)['cum_e']
                  .quantile(1.0 - trim_pct).reset_index(name='threshold')
    )
    per_user = per_user.merge(thresholds, on=['population', 'cost_date'], how='left')
    per_user['threshold'] = per_user['threshold'].fillna(np.inf)
    keep = per_user.loc[
        (per_user['cum_e'] == 0) | (per_user['cum_e'] <= per_user['threshold'])
    ][['population', 'cost_date', '__uid__']]
    return keep.copy()


def get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e):
    cfg = TRIM_CONFIG.get(population, {'method': 'cohort_trim', 'pct': 0.10})
    caps, trimmed = None, cohort_users
    if cfg['method'] == 'winsor':
        caps = compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=cfg['pct'])
    elif cfg['method'] == 'cohort_trim':
        trimmed = apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=cfg['pct'])
    return trimmed, caps


def sum_cum_at_idx(daily_user_cums, *, cohort_users, idx, caps=None):
    """Sum cumulative revenue across all users in cohort_users at day index idx."""
    grp_cols = ['population', 'cost_date']
    if idx < 0:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= idx].copy()
    if du.empty:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    per_user = (
        du.groupby(grp_cols + ['__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum')
    )
    per_user = cohort_users.merge(per_user, on=grp_cols + ['__uid__'], how='left')
    per_user['cum'] = per_user['cum'].fillna(0.0)
    if caps is not None:
        per_user = per_user.merge(caps, on=grp_cols + ['__uid__'], how='left')
        per_user['cap_e'] = per_user['cap_e'].fillna(np.inf)
        per_user['cum']   = np.minimum(per_user['cum'], per_user['cap_e'])
    sums = per_user.groupby(grp_cols, observed=True)['cum'].sum().reset_index(name='sum_cum')
    return sums


print('Trim + summation helpers defined.')


Trim + summation helpers defined.


In [8]:
# ══════════════════════════════════════════════════════════════
# ADAPTIVE CV ANALYSIS — persistent trim
# ══════════════════════════════════════════════════════════════

def patch_cv_adaptive(
    u_base, daily_user_cums, *,
    population, s, e, as_of_date,
    excluded_uids=None,
    lookback_cohorts=None,
    cv_threshold=None,
    cv_good_enough=None,
    max_remove_fraction=None,
    debug=True,
):
    # Resolve defaults at call time (after brand config applied)
    if lookback_cohorts is None:
        lookback_cohorts = LOOKBACK_COHORTS
    if cv_threshold is None:
        cv_threshold = CV_THRESHOLD
    if cv_good_enough is None:
        cv_good_enough = CV_GOOD_ENOUGH
    if max_remove_fraction is None:
        max_remove_fraction = MAX_REMOVE_FRACTION
    as_of_date   = pd.to_datetime(as_of_date).normalize()
    cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
    cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()

    cohort_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= cohort_start) &
        (u_base['cost_date'] <= cohort_end)
    ][['population', 'cost_date', '__uid__']].copy()

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    all_cohort_users = cohort_users.copy()
    n_users_in_cohort = int(cohort_users['__uid__'].nunique())

    if excluded_uids:
        cohort_users = cohort_users.loc[
            ~cohort_users['__uid__'].isin(excluded_uids)
        ].copy()

    n_users_after_prior = int(cohort_users['__uid__'].nunique())
    n_users_excluded_prior = n_users_in_cohort - n_users_after_prior

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    trimmed_users, caps = get_trimmed_cohort_and_caps(
        population, cohort_users, daily_user_cums, e
    )
    n_users_pre_trim  = n_users_after_prior
    n_users_post_trim = int(trimmed_users['__uid__'].nunique())

    newly_excluded = (
        set(cohort_users['__uid__'].unique()) - set(trimmed_users['__uid__'].unique())
    )

    denom_w = (
        trimmed_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                     .nunique().reset_index(name='N_users')
    )
    sum_s = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=s - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_s'})
    sum_e = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=e - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_e'})

    sum_e_all = sum_cum_at_idx(
        daily_user_cums, cohort_users=all_cohort_users, idx=e - 1, caps=None
    ).rename(columns={'sum_cum': 'sum_cum_e_all'})
    total_rev_before_trim = float(sum_e_all['sum_cum_e_all'].sum())

    patch = (
        denom_w
        .merge(sum_s, on=['population', 'cost_date'])
        .merge(sum_e, on=['population', 'cost_date'])
    )
    patch['ARPU_s']       = patch['sum_cum_s'] / patch['N_users']
    patch['ARPU_e']       = patch['sum_cum_e'] / patch['N_users']
    patch['growth_ratio'] = np.where(
        patch['ARPU_s'] > 0, patch['ARPU_e'] / patch['ARPU_s'], np.nan
    )

    _, _, cv_before = weighted_mean_std_cv(patch['growth_ratio'].values, patch['sum_cum_s'].values)

    mu_unw = np.nanmean(patch['growth_ratio'].values)
    patch['abs_dev'] = (patch['growth_ratio'] - mu_unw).abs()
    sorted_dates  = patch.sort_values('abs_dev', ascending=False)['cost_date'].tolist()
    max_removable = max(1, int(np.floor(len(patch) * max_remove_fraction)))

    removed   = []
    remaining = patch.copy()

    for candidate in sorted_dates:
        _, _, cv_now = weighted_mean_std_cv(
            remaining['growth_ratio'].values, remaining['sum_cum_s'].values
        )
        if np.isnan(cv_now) or cv_now <= cv_good_enough:
            break
        if len(removed) >= max_removable:
            break
        removed.append(candidate)
        remaining = remaining.loc[~remaining['cost_date'].isin(removed)]

    mean_a, _, cv_after = weighted_mean_std_cv(
        remaining['growth_ratio'].values, remaining['sum_cum_s'].values
    )
    flagged = (not np.isnan(cv_after)) and (cv_after > cv_threshold)
    total_rev_after_trim = float(patch['sum_cum_e'].sum())
    cfg = TRIM_CONFIG.get(population, {})

    if debug:
        flag_tag = f'  >>> FLAGGED (cv={cv_after:.4f} > {cv_threshold})' if flagged else ''
        print(
            f'  [{population}] {s}->{e}  '
            f'cv {cv_before:.4f}->{cv_after:.4f}  '
            f'removed={len(removed)}/{len(patch)}  '
            f'excl_prior={n_users_excluded_prior:,}  '
            f'pre/post={n_users_pre_trim:,}/{n_users_post_trim:,}  '
            f'newly_excl={len(newly_excluded):,}{flag_tag}'
        )

    stats = dict(
        population              = population,
        patch                   = f'{s}->{e}',
        cohort_start            = str(cohort_start),
        cohort_end              = str(cohort_end),
        n_cohort_dates_total    = int(len(patch)),
        n_cohort_dates_kept     = int(len(remaining)),
        n_users_excluded_prior  = n_users_excluded_prior,
        n_users_pre_trim        = n_users_pre_trim,
        n_users_post_trim       = n_users_post_trim,
        n_users_dropped_by_trim = n_users_pre_trim - n_users_post_trim,
        total_rev_before_trim   = total_rev_before_trim,
        total_rev_after_trim    = total_rev_after_trim,
        cv_before               = float(cv_before) if not np.isnan(cv_before) else None,
        cv_after                = float(cv_after)  if not np.isnan(cv_after)  else None,
        mean_after              = float(mean_a)    if not np.isnan(mean_a)    else None,
        flagged                 = bool(flagged),
        removed_dates           = removed,
        trim_method             = cfg.get('method', 'none'),
        trim_pct                = cfg.get('pct', 0),
    )
    return patch, stats, removed, flagged, newly_excluded


print('Adaptive CV function defined (persistent trim).')

# ══════════════════════════════════════════════════════════════
# ADDED: walk-forward backtest helpers (framework only)
# Production CV / trim / goal formulas are NOT changed.
#
# Patch-level "Marketing Goal" in this experiment =
#   mean_after from patch_cv_adaptive
#   = weighted mean of kept cohort growth_ratios (ARPU_e / ARPU_s)
# This is exactly the quantity that existing CV summarizes.
# Full day-level goal curves / organic are out of scope here.
# ══════════════════════════════════════════════════════════════

APE_NEAR_ZERO_GOAL = 1e-8  # ADDED: skip APE when |goal| below this


def generate_cutoff_dates(eval_as_of, *, freq_days, span_days, max_test_horizon, max_patch_e):
    """Historical as_of dates with room for training maturity + OOS test maturity.

    Latest cutoff T must allow test cohorts to reach day e by eval_as_of:
      T - e + max_test_horizon + e <= eval_as_of
      => T <= eval_as_of - max_test_horizon
    (plus 1-day slack).
    """
    eval_as_of = pd.Timestamp(eval_as_of).normalize()
    latest = eval_as_of - pd.Timedelta(days=int(max_test_horizon) + 1)
    earliest = latest - pd.Timedelta(days=int(span_days))
    dates = pd.date_range(earliest, latest, freq=f'{int(freq_days)}D')
    return [pd.Timestamp(d).normalize() for d in dates]


def growth_for_cost_date(u_base, daily_user_cums, *, population, s, e, cost_date):
    """Actual cohort growth_ratio for one cost_date (same winsor helper as production).

    Uses only users on that cost_date — no other dates enter the trim window.
    Documented framework choice: single-date winsor (not borrowing future/past dates).
    """
    cost_date = pd.to_datetime(cost_date).date() if not hasattr(cost_date, 'year') else cost_date
    cohort_users = u_base.loc[
        (u_base['population'] == population) & (u_base['cost_date'] == cost_date)
    ][['population', 'cost_date', '__uid__']].copy()
    if cohort_users.empty:
        return None
    trimmed_users, caps = get_trimmed_cohort_and_caps(
        population, cohort_users, daily_user_cums, e
    )
    if trimmed_users.empty:
        return None
    n_users = int(trimmed_users['__uid__'].nunique())
    sum_s = sum_cum_at_idx(daily_user_cums, cohort_users=trimmed_users, idx=s - 1, caps=caps)
    sum_e = sum_cum_at_idx(daily_user_cums, cohort_users=trimmed_users, idx=e - 1, caps=caps)
    if sum_s.empty or sum_e.empty:
        return None
    arpu_s = float(sum_s['sum_cum'].iloc[0]) / n_users if n_users else np.nan
    arpu_e = float(sum_e['sum_cum'].iloc[0]) / n_users if n_users else np.nan
    if not np.isfinite(arpu_s) or arpu_s <= 0 or not np.isfinite(arpu_e):
        return None
    return dict(
        cost_date=cost_date,
        n_users=n_users,
        ARPU_s=arpu_s,
        ARPU_e=arpu_e,
        actual_growth=arpu_e / arpu_s,
    )


def list_candidate_test_dates(u_base, *, population, training_end, n_wanted, e, eval_as_of):
    """Next eligible cost_dates after training_end that are mature by eval_as_of."""
    eval_as_of = pd.Timestamp(eval_as_of).normalize()
    training_end = pd.to_datetime(training_end).date()
    max_cost = (eval_as_of - pd.Timedelta(days=e)).date()
    dates = (
        u_base.loc[
            (u_base['population'] == population)
            & (u_base['cost_date'] > training_end)
            & (u_base['cost_date'] <= max_cost),
            'cost_date'
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    return dates[: int(n_wanted)]


def summarize_errors(actuals, goal):
    """Aggregate OOS errors; APE skipped when |goal| ~ 0."""
    actuals = np.asarray(actuals, dtype=float)
    actuals = actuals[np.isfinite(actuals)]
    out = dict(
        n_test_cohorts=int(actuals.size),
        test_mae=None, test_median_ae=None,
        test_mape=None, test_median_ape=None,
        test_rmse=None, test_bias=None,
    )
    if actuals.size == 0 or goal is None or not np.isfinite(goal):
        return out
    err = actuals - float(goal)
    ae = np.abs(err)
    out['test_mae'] = float(np.mean(ae))
    out['test_median_ae'] = float(np.median(ae))
    out['test_rmse'] = float(np.sqrt(np.mean(err ** 2)))
    out['test_bias'] = float(np.mean(err))
    if abs(float(goal)) > APE_NEAR_ZERO_GOAL:
        ape = ae / abs(float(goal))
        out['test_mape'] = float(np.mean(ape))
        out['test_median_ape'] = float(np.median(ape))
    return out


def run_walkforward_backtest(
    brand_key,
    users_df,
    revenue_df,
    *,
    cutoff_dates,
    eval_as_of,
    test_horizons,
    debug=False,
):
    """For each historical cutoff T: run existing patch CV path, freeze mean_after, score OOS."""
    cfg = BRAND_CONFIGS[brand_key]
    apply_brand_globals(cfg)
    brand = cfg['brand']
    rows = []
    point_rows = []  # per test cohort date

    # Precompute cums once (data is historical; eligibility enforced via as_of in patch_cv)
    pops = list(cfg['populations'])
    u_pop, daily_pop = build_user_revenue_cums(
        users_df.loc[users_df['population'].isin(pops)].copy(),
        revenue_df,
        max_day=365,
    )
    users_blend = users_df.copy()
    users_blend['population'] = 'Blended'
    u_blend, daily_blend = build_user_revenue_cums(users_blend, revenue_df, max_day=365)

    for T in cutoff_dates:
        T = pd.Timestamp(T).normalize()
        if debug:
            print(f'\n[{brand}] cutoff as_of={T.date()}')

        # Existing methodology path at time T (per-pop + blended)
        global POPULATIONS
        POPULATIONS = list(cfg['populations'])
        cv_pop, _ = build_all_populations(
            u_pop, daily_pop, as_of_date=T, extrapolate_tail=False, debug=debug,
        )
        POPULATIONS = ['Blended']
        try:
            cv_blend, _ = build_all_populations(
                u_blend, daily_blend, as_of_date=T, extrapolate_tail=False, debug=debug,
            )
        finally:
            POPULATIONS = list(cfg['populations'])

        cv_t = pd.concat(
            [x for x in [cv_pop, cv_blend] if x is not None and not x.empty],
            ignore_index=True,
        )
        if cv_t.empty:
            continue

        for _, st in cv_t.iterrows():
            pop = st['population']
            patch = st['patch']
            s, e = map(int, patch.split('->'))
            goal = st.get('mean_after')
            training_start = st.get('cohort_start')
            training_end = st.get('cohort_end')
            u_base = u_blend if pop == 'Blended' else u_pop
            daily = daily_blend if pop == 'Blended' else daily_pop

            for h in test_horizons:
                test_dates = list_candidate_test_dates(
                    u_base, population=pop, training_end=training_end,
                    n_wanted=h, e=e, eval_as_of=eval_as_of,
                )
                actuals = []
                for d in test_dates:
                    g = growth_for_cost_date(
                        u_base, daily, population=pop, s=s, e=e, cost_date=d,
                    )
                    if g is None:
                        continue
                    actuals.append(g['actual_growth'])
                    if goal is not None and np.isfinite(goal):
                        ae = abs(g['actual_growth'] - float(goal))
                        ape = (ae / abs(float(goal))) if abs(float(goal)) > APE_NEAR_ZERO_GOAL else np.nan
                        point_rows.append({
                            'brand': brand,
                            'population': pop,
                            'patch': patch,
                            'cutoff_as_of': str(T.date()),
                            'training_start': training_start,
                            'training_end': training_end,
                            'goal': float(goal) if goal is not None and np.isfinite(goal) else None,
                            'cv_after': st.get('cv_after'),
                            'flagged': st.get('flagged'),
                            'test_horizon': int(h),
                            'test_cost_date': str(g['cost_date']),
                            'actual_growth': g['actual_growth'],
                            'abs_error': ae,
                            'ape': ape if np.isfinite(ape) else None,
                            'signed_error': g['actual_growth'] - float(goal),
                            'n_test_users': g['n_users'],
                        })

                err = summarize_errors(actuals, goal)
                # insufficient future data → report, do not fill
                rows.append({
                    'brand': brand,
                    'population': pop,
                    'patch': patch,
                    'cutoff_as_of': str(T.date()),
                    'training_start': training_start,
                    'training_end': training_end,
                    'n_training_cohorts': st.get('n_cohort_dates_total'),
                    'n_training_cohorts_kept': st.get('n_cohort_dates_kept'),
                    'n_training_users': st.get('n_users_post_trim'),
                    'goal': float(goal) if goal is not None and np.isfinite(float(goal) if goal is not None else np.nan) else None,
                    'cv_before': st.get('cv_before'),
                    'cv_after': st.get('cv_after'),
                    'flagged_using_existing_logic': bool(st.get('flagged')),
                    'cv_threshold_used': float(CV_THRESHOLD),
                    'test_horizon': int(h),
                    'n_test_dates_available': len(test_dates),
                    'insufficient_test_data': len(actuals) < int(h),
                    **err,
                })

    return pd.DataFrame(rows), pd.DataFrame(point_rows)


print('Walk-forward backtest helpers defined.')


Adaptive CV function defined (persistent trim).
Walk-forward backtest helpers defined.


In [9]:
# ══════════════════════════════════════════════════════════════
# PERSISTENT TRIM CURVE BUILDER (two-pass)
# ══════════════════════════════════════════════════════════════

def build_curve(
    u_base, daily_user_cums, *,
    population, as_of_date, debug=True,
):
    """Build ARPU curve with persistent trim — excluded users carry forward.
    Two-pass: first pass accumulates all exclusions unconditionally,
    second pass builds step ratios using frozen snapshots."""
    as_of_date = pd.to_datetime(as_of_date).normalize()

    cv_rows      = []
    effective    = []
    excluded_uids = set()

    # ── First pass: determine valid patches & accumulate exclusions ──
    for (s, e) in PATCHES:
        patch, stats, removed, flagged, newly_excluded = patch_cv_adaptive(
            u_base, daily_user_cums,
            population=population, s=s, e=e,
            as_of_date=as_of_date,
            excluded_uids=excluded_uids,
            debug=debug,
        )
        excluded_uids |= newly_excluded

        if not stats:
            continue

        min_cohort_dates = globals().get('MIN_COHORT_DATES', 1)
        if stats.get('n_cohort_dates_total', 0) < min_cohort_dates:
            if debug:
                print(f'  [{population}] {s}->{e}  insufficient data '
                      f'({stats["n_cohort_dates_total"]} < {min_cohort_dates}) — skipping')
            continue

        cv_rows.append(stats)
        effective.append({
            's': s, 'e': e,
            'removed_dates': removed,
            'excluded_snapshot': frozenset(excluded_uids),
        })

    if not effective:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    if debug:
        print(f'  Total unique users excluded across all patches: {len(excluded_uids):,}')

    # ── Second pass: build step ratios using frozen snapshots ──
    step_rows = []
    for ep in effective:
        s, e      = ep['s'], ep['e']
        bad_dates = set(ep['removed_dates'])
        excl      = ep['excluded_snapshot']
        start_k   = 2 if s == 1 else (s + 1)

        cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=e + (LOOKBACK_COHORTS - 1))).date()

        cohort_users = u_base.loc[
            (u_base['population'] == population) &
            (u_base['cost_date'] >= cohort_start) &
            (u_base['cost_date'] <= cohort_end)
        ][['population', 'cost_date', '__uid__']].copy()

        if bad_dates:
            cohort_users = cohort_users.loc[~cohort_users['cost_date'].isin(bad_dates)].copy()
        if excl:
            cohort_users = cohort_users.loc[~cohort_users['__uid__'].isin(excl)].copy()
        if cohort_users.empty:
            continue

        _, caps = get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e)
        denom_w = (
            cohort_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                        .nunique().reset_index(name='N_users')
        )

        for k in range(start_k, e + 1):
            sum_prev = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 2, caps=caps
            ).rename(columns={'sum_cum': 'sum_prev'})
            sum_curr = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 1, caps=caps
            ).rename(columns={'sum_cum': 'sum_curr'})
            tmp = (
                denom_w
                .merge(sum_prev, on=['population', 'cost_date'])
                .merge(sum_curr, on=['population', 'cost_date'])
            )
            tmp['ARPU_prev']  = tmp['sum_prev'] / tmp['N_users']
            tmp['ARPU_curr']  = tmp['sum_curr'] / tmp['N_users']
            tmp['step_ratio'] = np.where(
                tmp['ARPU_prev'] > 0, tmp['ARPU_curr'] / tmp['ARPU_prev'], np.nan
            )
            mean_step, _, _ = weighted_mean_std_cv(
                tmp['step_ratio'].values, tmp['sum_prev'].values
            )
            step_rows.append({
                'population':      population,
                'day':             int(k),
                'growth_step':     float(mean_step),
                'effective_patch': f'{s}->{e}',
            })

    if not step_rows:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    step_df = pd.DataFrame(step_rows).sort_values('day').reset_index(drop=True)

    # ── Base ARPU from first effective patch ──
    first      = effective[0]
    fs, fe     = first['s'], first['e']
    excl_first = first['excluded_snapshot']
    base_end   = (as_of_date - pd.Timedelta(days=fe)).date()
    base_start = (as_of_date - pd.Timedelta(days=fe + (LOOKBACK_COHORTS - 1))).date()

    base_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= base_start) &
        (u_base['cost_date'] <= base_end)
    ][['population', 'cost_date', '__uid__']].copy()
    bad_first = set(first['removed_dates'])
    if bad_first:
        base_users = base_users.loc[~base_users['cost_date'].isin(bad_first)].copy()
    if excl_first:
        base_users = base_users.loc[~base_users['__uid__'].isin(excl_first)].copy()

    _, base_caps = get_trimmed_cohort_and_caps(population, base_users, daily_user_cums, fe)
    denom_base = (
        base_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                  .nunique().reset_index(name='N_users')
    )
    start_idx = 0 if fs == 1 else (fs - 1)
    sum_day1 = sum_cum_at_idx(
        daily_user_cums, cohort_users=base_users, idx=start_idx, caps=base_caps
    ).rename(columns={'sum_cum': 'sum_day1'})
    base = denom_base.merge(sum_day1, on=['population', 'cost_date'], how='inner')
    pooled_arpu_1 = (
        base['sum_day1'].sum() / base['N_users'].sum()
        if base['N_users'].sum() > 0 else 0.0
    )

    start_day = 1 if fs == 1 else fs
    out_rows  = [{'population': population, 'day': start_day,
                  'ARPU_nominal': float(pooled_arpu_1),
                  'growth_step':  np.nan,
                  'effective_patch': f'{fs}->{fe}'}]
    arpu = float(pooled_arpu_1)
    for _, row in step_df.iterrows():
        g = row['growth_step']
        if not np.isfinite(g):
            continue
        arpu *= float(g)
        out_rows.append({'population': population, 'day': int(row['day']),
                         'ARPU_nominal': float(arpu),
                         'growth_step':  float(g),
                         'effective_patch': row['effective_patch']})

    curve = pd.DataFrame(out_rows).sort_values('day').reset_index(drop=True)
    all_days = pd.DataFrame({'day': range(start_day, 366)})
    curve    = all_days.merge(curve, on='day', how='left')
    curve['population']      = population
    curve['effective_patch'] = curve['effective_patch'].ffill()
    curve['ARPU_nominal']    = curve['ARPU_nominal'].interpolate(
        method='linear', limit_area='inside'
    )
    curve = curve.dropna(subset=['ARPU_nominal']).reset_index(drop=True)
    curve['is_extrapolated'] = False

    return pd.DataFrame(cv_rows), curve


def extrapolate_curve_tail(curve, *, up_to_day=365, tail_days=30, debug=True):
    """Extend curve to up_to_day using geometric mean of last tail_days daily growth steps."""
    if curve.empty:
        return curve

    population    = curve['population'].iloc[0]
    last_real_day = int(curve['day'].max())
    if last_real_day >= up_to_day:
        return curve

    real = curve.loc[~curve['is_extrapolated']].sort_values('day')
    tail = real.tail(tail_days)
    if len(tail) < 2:
        if debug:
            print(f'  [{population}] Not enough tail data ({len(tail)}) to extrapolate.')
        return curve

    arpu_vals = tail['ARPU_nominal'].values
    ratios    = arpu_vals[1:] / arpu_vals[:-1]
    ratios    = ratios[np.isfinite(ratios) & (ratios > 0)]
    if len(ratios) == 0:
        return curve

    avg_daily_growth = float(np.exp(np.mean(np.log(ratios))))

    if debug:
        print(
            f'  [{population}] Extrapolating D{last_real_day + 1}→D{up_to_day}  '
            f'rate={avg_daily_growth:.6f}/day  (geom. mean of last {len(ratios)} daily ratios)'
        )

    last_arpu  = float(curve.loc[curve['day'] == last_real_day, 'ARPU_nominal'].iloc[0])
    last_patch = curve.loc[curve['day'] == last_real_day, 'effective_patch'].iloc[0]
    tag_patch  = f'{last_patch} (extrapolated)'

    new_rows = []
    arpu = last_arpu
    for d in range(last_real_day + 1, up_to_day + 1):
        arpu *= avg_daily_growth
        new_rows.append({
            'population':      population,
            'day':             d,
            'ARPU_nominal':    float(arpu),
            'growth_step':     avg_daily_growth,
            'effective_patch': tag_patch,
            'is_extrapolated': True,
        })

    return pd.concat([curve, pd.DataFrame(new_rows)], ignore_index=True)


def build_all_populations(
    u_base, daily_user_cums, *,
    as_of_date, debug=True,
    extrapolate_tail=False, tail_days=30, extrapolate_up_to=365,
):
    all_cv_rows = []
    all_curves  = []
    for pop in POPULATIONS:
        if debug:
            print(f'\n{"=" * 55}')
            print(f'POPULATION: {pop}')
            print(f'{"=" * 55}')
        cv_df, curve = build_curve(
            u_base, daily_user_cums,
            population=pop, as_of_date=as_of_date, debug=debug,
        )
        if not cv_df.empty:
            all_cv_rows.append(cv_df)
        if not curve.empty and extrapolate_tail:
            curve = extrapolate_curve_tail(
                curve, up_to_day=extrapolate_up_to, tail_days=tail_days, debug=debug,
            )
        if not curve.empty:
            all_curves.append(curve)
    cv_out    = pd.concat(all_cv_rows, ignore_index=True) if all_cv_rows else pd.DataFrame()
    curve_out = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
    return cv_out, curve_out

print('Persistent trim curve builder defined.')


Persistent trim curve builder defined.


In [10]:
# Organic skipped in cv_oos_backtest (patch growth goal only).
print('Organic helpers skipped (cv_oos_backtest).')


Organic helpers skipped (cv_oos_backtest).


In [11]:
# ══════════════════════════════════════════════════════════════
# PART 4 helpers — Goal construction + brand pipeline
# Per-population: adjusted = (ARPU_day / ARPU_horizon) × (1 − organic_share_at_horizon)
# Blended:        adjusted = raw_goal_ratio (no organic adjustment)
# ──
# Organic share rule: for every day inside a horizon, use the share measured AT
# the horizon endpoint (not the share at the day's nearest checkpoint).
# RP: pin lookup to min(horizon, ORGANIC_SHARE_CAP_HORIZON) when cap is set.
# LS: no cap (cap is None).
# ══════════════════════════════════════════════════════════════

def build_goals(curve_df, organic_df, populations, goal_horizons=None,
                organic_share_cap_horizon=None):
    if goal_horizons is None:
        goal_horizons = GOAL_HORIZONS

    has_scope = 'scope' in organic_df.columns
    available_scopes = set(organic_df['scope'].unique()) if has_scope else {'all'}

    org_lookup = {}
    scope_col = 'scope' if has_scope else None
    endpoint = organic_df.loc[organic_df['checkpoint_day'] == organic_df['goal_horizon']]
    if scope_col:
        for scope, grp in endpoint.groupby(scope_col):
            org_lookup[scope] = grp.set_index('goal_horizon')['organic_share_pct'].to_dict()
    else:
        org_lookup['all'] = endpoint.set_index('goal_horizon')['organic_share_pct'].to_dict()

    def pop_to_scope(pop):
        if pop == 'App' and 'app' in available_scopes:
            return 'app'
        if 'non_app' in available_scopes:
            return 'non_app'
        return next(iter(available_scopes))

    rows = []
    all_pops = list(populations) + ['Blended']

    for pop in all_pops:
        is_blended = (pop == 'Blended')

        pop_curve = curve_df.loc[curve_df['population'] == pop].copy()
        if pop_curve.empty:
            continue
        pop_curve = pop_curve.drop_duplicates(subset='day').set_index('day')

        for horizon in goal_horizons:
            if horizon not in pop_curve.index:
                print(f'[{pop}] Day {horizon} missing from curve — skipping horizon.')
                continue
            arpu_horizon = pop_curve.loc[horizon, 'ARPU_nominal']
            if not np.isfinite(arpu_horizon) or arpu_horizon == 0:
                continue

            if is_blended:
                org_share = 0.0
            else:
                scope = pop_to_scope(pop)
                if organic_share_cap_horizon is not None:
                    lookup_horizon = min(horizon, organic_share_cap_horizon)
                else:
                    lookup_horizon = horizon
                org_share = org_lookup.get(scope, {}).get(lookup_horizon, np.nan)

            for day in range(1, horizon + 1):
                if day not in pop_curve.index:
                    continue
                arpu_day  = pop_curve.loc[day, 'ARPU_nominal']
                eff_patch = pop_curve.loc[day, 'effective_patch'] if 'effective_patch' in pop_curve.columns else ''
                is_extrap = bool(pop_curve.loc[day, 'is_extrapolated']) if 'is_extrapolated' in pop_curve.columns else False
                raw_goal  = arpu_day / arpu_horizon

                if np.isfinite(org_share):
                    adj_goal = raw_goal * (1 - org_share)
                else:
                    adj_goal = np.nan

                rows.append(dict(
                    population           = pop,
                    goal_horizon         = horizon,
                    day                  = day,
                    ARPU_nominal         = float(arpu_day),
                    ARPU_at_horizon      = float(arpu_horizon),
                    raw_goal_ratio       = float(raw_goal),
                    organic_share        = float(org_share) if np.isfinite(org_share) else None,
                    adjusted_goal_ratio  = float(adj_goal)  if np.isfinite(adj_goal)  else None,
                    effective_patch      = eff_patch,
                    is_extrapolated      = is_extrap,
                ))

    return pd.DataFrame(rows)


def _banner(title: str):
    print('\n' + '─' * 60)
    print(title)
    print('─' * 60)


def monitor_show_users(brand, users_df):
    """After SQL load: population counts + date span."""
    _banner(f'[{brand}] MONITOR · DATA LOAD')
    print(f'users_df rows:   {len(users_df):,}')
    print('population counts:')
    print(users_df['population'].value_counts().to_string())
    cd = pd.to_datetime(users_df['cost_date'], errors='coerce')
    print(f"cost_date span: {cd.min().date()} → {cd.max().date()}")
    if 'scope' in users_df.columns:
        print('\nscope × bucket:')
        print(users_df.groupby(['scope', 'bucket'], observed=True).size().to_string())


def monitor_show_cv_curve(brand, stage, cv_df, curve_df, populations):
    """After ARPU curve build: CV table + milestone ARPU."""
    _banner(f'[{brand}] MONITOR · {stage}')
    if cv_df is not None and not cv_df.empty:
        display_cols = [
            'population', 'patch', 'n_dates_before', 'n_dates_after',
            'cv_before', 'cv_after', 'removed_dates',
            'total_rev_before_trim', 'total_rev_after_trim',
            'flagged', 'trim_method',
        ]
        display_cols = [c for c in display_cols if c in cv_df.columns]
        print('CV summary:')
        print(cv_df[display_cols].to_string(index=False))
        if 'flagged' in cv_df.columns:
            flagged = cv_df.loc[cv_df['flagged'] == True]
            if not flagged.empty:
                print('\n>>> FLAGGED PATCHES:')
                for _, r in flagged.iterrows():
                    patch = r['patch'] if 'patch' in r.index else '?'
                    print(f"    {r.get('population', '?')} / {patch}  CV={r.get('cv_after', float('nan')):.4f}")
            else:
                print('\nNo flagged patches in this stage.')
    else:
        print('CV summary: (empty)')

    if curve_df is not None and not curve_df.empty:
        milestones = list(MONITOR_PREVIEW_DAYS)
        for pop in populations:
            sub = curve_df.loc[curve_df['population'] == pop]
            if sub.empty:
                continue
            cols = [c for c in ['day', 'ARPU_nominal', 'effective_patch', 'is_extrapolated'] if c in sub.columns]
            ms = sub.loc[sub['day'].isin(milestones)][cols]
            print(f'\nARPU curve milestones — {pop}  (n_days={sub["day"].nunique()})')
            print(ms.to_string(index=False))
            if 'is_extrapolated' in sub.columns and sub['is_extrapolated'].any():
                n_ex = int(sub['is_extrapolated'].sum())
                last_real = int(sub.loc[~sub['is_extrapolated'], 'day'].max()) if (~sub['is_extrapolated']).any() else None
                print(f'  extrapolated days: {n_ex:,}  last real day: {last_real}')
    else:
        print('ARPU curve: (empty)')


def monitor_show_organic(brand, organic_df):
    """After organic share computation."""
    _banner(f'[{brand}] MONITOR · ORGANIC SHARE')
    if organic_df is None or organic_df.empty:
        print('organic_df: (empty)')
        return
    print(f'rows: {len(organic_df):,}')
    # endpoint only (checkpoint_day == goal_horizon) — what goals actually use
    if 'checkpoint_day' in organic_df.columns and 'goal_horizon' in organic_df.columns:
        ep = organic_df.loc[organic_df['checkpoint_day'] == organic_df['goal_horizon']].copy()
        cols = [c for c in ['scope', 'goal_horizon', 'organic_share_pct', 'users_org', 'users_acq'] if c in ep.columns]
        for scope in ep['scope'].unique() if 'scope' in ep.columns else [None]:
            sub = ep if scope is None else ep.loc[ep['scope'] == scope]
            title = f'endpoint shares — scope={scope}' if scope is not None else 'endpoint shares'
            print(f'\n{title}:')
            print(sub[cols].to_string(index=False))
    else:
        print(organic_df.head(20).to_string(index=False))


def monitor_show_goals(brand, goals_df, populations):
    """After goal construction: sample horizons × days."""
    _banner(f'[{brand}] MONITOR · GOALS')
    if goals_df is None or goals_df.empty:
        print('goals_df: (empty)')
        return
    print(f'total goal rows: {len(goals_df):,}')
    print('counts by population:')
    print(goals_df.groupby('population', observed=True).size().to_string())

    preview_days = list(MONITOR_PREVIEW_DAYS)
    preview_horizons = list(MONITOR_PREVIEW_HORIZONS)
    preview_cols = [
        c for c in [
            'population', 'goal_horizon', 'day',
            'raw_goal_ratio', 'organic_share', 'adjusted_goal_ratio',
            'ARPU_nominal', 'is_extrapolated',
        ] if c in goals_df.columns
    ]
    for pop in list(populations) + ['Blended']:
        for horizon in preview_horizons:
            sub = goals_df.loc[
                (goals_df['population'] == pop) &
                (goals_df['goal_horizon'] == horizon) &
                (goals_df['day'].isin(preview_days))
            ]
            if sub.empty:
                continue
            print(f'\n── {pop} │ horizon D{horizon} ──')
            print(sub[preview_cols].to_string(index=False))


def run_brand_pipeline(cfg, users_df, revenue_df, as_of_date=AS_OF_DATE, monitor=None):
    """Parts 1-4 for one brand. Returns dict of frames with brand column.

    monitor=True  -> print intermediate tables after every part (for Colab monitoring)
    monitor=False -> banners only
    monitor=None  -> use global MONITOR_STEPS
    """
    global POPULATIONS  # reassigned briefly for Blended pass; must be declared before first use

    if monitor is None:
        monitor = bool(globals().get('MONITOR_STEPS', True))

    apply_brand_globals(cfg)
    brand = cfg['brand']

    if monitor:
        monitor_show_users(brand, users_df)

    # ── PART 1 — per-population ARPU curves ──
    _banner(f'[{brand}] PART 1/4 — per-population ARPU curves')
    print(f'[{brand}] Building base cums + adaptive CV patches (debug lines below)...')
    u_pop, daily_pop = build_user_revenue_cums(
        users_df.loc[users_df['population'].isin(POPULATIONS)].copy(),
        revenue_df,
        max_day=365,
    )
    print(f'[{brand}] base users in ARPU pops: {len(u_pop):,}  daily_user rows: {len(daily_pop):,}')

    cv_pop_df, curve_pop_df = build_all_populations(
        u_pop, daily_pop,
        as_of_date=as_of_date,
        extrapolate_tail=EXTRAPOLATE_TAIL,
        tail_days=EXTRAPOLATION_TAIL_DAYS,
        extrapolate_up_to=365,
        debug=True,
    )
    if monitor:
        monitor_show_cv_curve(brand, 'PART 1 done — per-pop CV + curve', cv_pop_df, curve_pop_df, POPULATIONS)

    # ── PART 2 — Blended ──
    _banner(f'[{brand}] PART 2/4 — Blended ARPU curve')
    print(f'[{brand}] Building BLENDED...')
    users_blend = users_df.copy()
    users_blend['population'] = 'Blended'
    u_blend, daily_blend = build_user_revenue_cums(
        users_blend, revenue_df, max_day=365
    )
    print(f'[{brand}] blended users: {len(u_blend):,}')
    # Temporarily set POPULATIONS so build_all_populations iterates Blended only
    _BLENDED_POPS_BACKUP = list(POPULATIONS)
    POPULATIONS = ['Blended']
    try:
        cv_blend_df, curve_blend_df = build_all_populations(
            u_blend, daily_blend,
            as_of_date=as_of_date,
            extrapolate_tail=EXTRAPOLATE_TAIL,
            tail_days=EXTRAPOLATION_TAIL_DAYS,
            extrapolate_up_to=365,
            debug=True,
        )
    finally:
        POPULATIONS = _BLENDED_POPS_BACKUP

    if monitor:
        monitor_show_cv_curve(brand, 'PART 2 done — Blended CV + curve', cv_blend_df, curve_blend_df, ['Blended'])

    cv_df    = pd.concat([cv_pop_df,    cv_blend_df],    ignore_index=True)
    curve_df = pd.concat([curve_pop_df, curve_blend_df], ignore_index=True)

    # ── PART 3 — organic share ──
    _banner(f'[{brand}] PART 3/4 — organic share')
    org_label = f'{ORGANIC_TRIM_METHOD}_{round(ORGANIC_TRIM_PCT * 100):g}pct'
    org_config = [{'method': ORGANIC_TRIM_METHOD, 'pct': ORGANIC_TRIM_PCT, 'label': org_label}]
    print(f'[{brand}] Computing organic share [{org_label}] for horizons {GOAL_HORIZONS}...')
    organic_full = organic_share_cohort_progression(
        users_df, revenue_df,
        as_of_date=as_of_date,
        goal_horizons=GOAL_HORIZONS,
        checkpoints=CHECKPOINTS,
        lookback_cohorts=LOOKBACK_COHORTS,
        positive_amount_only=True,
        trim_configs=org_config,
        persistent_trim=True,
    )
    organic_df = organic_full[['scope', 'goal_horizon', 'cohort_start', 'cohort_end',
                               'checkpoint_day',
                               f'organic_share_pct_{org_label}',
                               f'users_org_{org_label}',
                               f'users_acq_{org_label}']].copy()
    organic_df = organic_df.rename(columns={
        f'organic_share_pct_{org_label}': 'organic_share_pct',
        f'users_org_{org_label}': 'users_org',
        f'users_acq_{org_label}': 'users_acq',
    })
    if monitor:
        monitor_show_organic(brand, organic_df)
    else:
        print(f'[{brand}] organic rows: {len(organic_df):,}')

    # ── PART 4 — goals ──
    _banner(f'[{brand}] PART 4/4 — goal construction')
    goals_df = build_goals(
        curve_df, organic_df, POPULATIONS,
        goal_horizons=GOAL_HORIZONS,
        organic_share_cap_horizon=ORGANIC_SHARE_CAP_HORIZON,
    )
    if monitor:
        monitor_show_goals(brand, goals_df, POPULATIONS)
    else:
        print(f'[{brand}] goals rows: {len(goals_df):,}')

    # stamp brand
    for df in (cv_df, curve_df, organic_df, goals_df):
        if not df.empty:
            df.insert(0, 'brand', brand)

    print(f'\n[{brand}] COMPLETE — goals {len(goals_df):,} | curve {len(curve_df):,} | organic {len(organic_df):,} | cv {len(cv_df):,}')
    return {
        'cv': cv_df,
        'curve': curve_df,
        'organic': organic_df,
        'goals': goals_df,
        'populations': list(cfg['populations']),
    }

print('run_brand_pipeline + monitors ready.')
print('Set MONITOR_STEPS=True in config cell to print intermediate tables after each part.')


run_brand_pipeline + monitors ready.
Set MONITOR_STEPS=True in config cell to print intermediate tables after each part.


In [ ]:
# ══════════════════════════════════════════════════════════════
# WALK-FORWARD BACKTEST RUN
# ══════════════════════════════════════════════════════════════

eval_as_of = AS_OF_DATE  # pinned evaluation "today" for maturity of OOS cohorts
max_e = max(e for _, e in PATCHES)
cutoff_dates = generate_cutoff_dates(
    eval_as_of,
    freq_days=CUTOFF_FREQ_DAYS,
    span_days=CUTOFF_SPAN_DAYS,
    max_test_horizon=max(TEST_HORIZONS),
    max_patch_e=max_e,
)
print(f'EVAL_AS_OF (maturity ceiling) = {eval_as_of.date()}')
print(f'Cutoffs: n={len(cutoff_dates)}  first={cutoff_dates[0].date()}  last={cutoff_dates[-1].date()}')
print(f'TEST_HORIZONS = {TEST_HORIZONS}  CUTOFF_FREQ_DAYS = {CUTOFF_FREQ_DAYS}')
print('Look-ahead controls:')
print('  - training window via patch_cv_adaptive(as_of=T) only')
print('  - test dates strictly > training_end and mature by EVAL_AS_OF')
print('  - no future dates in trim/CV/goal at time T')

wf_parts = []
pt_parts = []
brand_cache = {}

for brand_key in RUN_BRANDS:
    cfg = BRAND_CONFIGS[brand_key]
    # Deep SQL floor: history for earliest cutoff + longest patch + lookback + slack
    # load_brand_tables uses LOOKBACK_COHORTS; temporarily widen floor via as_of span
    users_df, revenue_df = load_brand_tables(cfg, as_of_date=eval_as_of)
    # Extra safety: drop any revenue/users after eval_as_of (should already be constrained)
    if 'date' in revenue_df.columns:
        revenue_df = revenue_df.loc[
            pd.to_datetime(revenue_df['date'], errors='coerce') <= eval_as_of
        ].copy()
    brand_cache[brand_key] = (users_df, revenue_df)
    print(f'\n=== BACKTEST {brand_key}  users={len(users_df):,} rev_rows={len(revenue_df):,} ===')
    wf_b, pt_b = run_walkforward_backtest(
        brand_key, users_df, revenue_df,
        cutoff_dates=cutoff_dates,
        eval_as_of=eval_as_of,
        test_horizons=TEST_HORIZONS,
        debug=False,
    )
    if not wf_b.empty:
        wf_parts.append(wf_b)
    if not pt_b.empty:
        pt_parts.append(pt_b)
    print(f'  window-rows={len(wf_b):,}  point-rows={len(pt_b):,}')

wf_df = pd.concat(wf_parts, ignore_index=True) if wf_parts else pd.DataFrame()
pt_df = pd.concat(pt_parts, ignore_index=True) if pt_parts else pd.DataFrame()

# Primary analysis horizon (configurable)
PRIMARY_H = PRIMARY_TEST_HORIZON
wf_primary = wf_df.loc[wf_df['test_horizon'] == PRIMARY_H].copy() if not wf_df.empty else wf_df

print('\n' + '=' * 60)
print('WALK-FORWARD COMPLETE')
print(f'  experiment          = {EXPERIMENT_TAG}')
print(f'  window rows         = {len(wf_df):,}')
print(f'  point rows          = {len(pt_df):,}')
print(f'  primary test_horizon= {PRIMARY_H}')
print(f'  primary rows        = {len(wf_primary):,}')
if not wf_df.empty:
    print('  insufficient_test_data rate:',
          float(wf_df['insufficient_test_data'].mean()))
print('=' * 60)


EVAL_AS_OF (maturity ceiling) = 2026-08-03
Cutoffs: n=13  first=2026-01-04  last=2026-06-21
TEST_HORIZONS = [7, 14, 30]  CUTOFF_FREQ_DAYS = 14
Look-ahead controls:
  - training window via patch_cv_adaptive(as_of=T) only
  - test dates strictly > training_end and mature by EVAL_AS_OF
  - no future dates in trim/CV/goal at time T
[realprize] SQL floor date: 2024-10-22  (AS_OF_DATE = 2026-08-03)
Downloading: 100%|██████████|
Downloading: 100%|██████████|
[realprize] users_df:   1,771,531 rows
[realprize] revenue_df: 2,531,238 rows
population
Affiliate    513504
App          360450
Web          348403
Organic      285358
PPC          263816

=== BACKTEST realprize  users=1,771,531 rev_rows=2,491,553 ===

BRAND: realprize
  populations = ['Web', 'App', 'Affiliate']
  CV threshold = 0.15  good_enough = 0.1
  min_cohort_dates = 1
  extrapolate_tail = False
  organic_share_cap_horizon = 120


In [ ]:
# ══════════════════════════════════════════════════════════════
# ANALYSIS — Does CV predict OOS goal error?
# ══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

assert not wf_df.empty, 'wf_df empty — run backtest cell first'

# Use rows with at least 1 OOS observation
W = wf_primary.loc[wf_primary['n_test_cohorts'] > 0].copy()
print(f'Primary horizon H={PRIMARY_H}: usable windows = {len(W)} / {len(wf_primary)}')

ERR = 'test_median_ae'  # primary error measure (robust to outliers); also report MAE/RMSE

# ── 4) Correlations CV vs error ──
print('\n=== CV vs future prediction error (correlations) ===')

def corr_block(df, label):
    d = df.dropna(subset=['cv_after', ERR])
    if len(d) < 5:
        print(f'{label}: insufficient n={len(d)}')
        return
    pear = d['cv_after'].corr(d[ERR], method='pearson')
    spear = d['cv_after'].corr(d[ERR], method='spearman')
    print(f'{label}: n={len(d)}  Pearson({ERR})={pear:.3f}  Spearman={spear:.3f}')
    for col in ['test_mae', 'test_rmse', 'test_median_ape', 'test_mape']:
        if col in d.columns and d[col].notna().sum() >= 5:
            print(f'    vs {col}: pearson={d["cv_after"].corr(d[col], method="pearson"):.3f}  '
                  f'spearman={d["cv_after"].corr(d[col], method="spearman"):.3f}')

corr_block(W, 'OVERALL')
for patch in [f'{a}->{b}' for a, b in PATCHES]:
    corr_block(W.loc[W['patch'] == patch], f'patch {patch}')

# ── 5) Threshold 0.15 ──
print('\n=== Existing ~15% threshold: CV<=0.15 vs CV>0.15 ===')

def threshold_compare(df, label):
    d = df.dropna(subset=['cv_after', ERR]).copy()
    if d.empty:
        print(label, '(empty)')
        return None
    d['cv_gt_15'] = d['cv_after'] > 0.15
    rows = []
    for flag, name in [(False, 'CV<=0.15'), (True, 'CV>0.15')]:
        sub = d.loc[d['cv_gt_15'] == flag]
        rows.append({
            'group': name,
            'n': len(sub),
            'median_error': sub[ERR].median() if len(sub) else None,
            'mean_error': sub[ERR].mean() if len(sub) else None,
            'p75_error': sub[ERR].quantile(0.75) if len(sub) else None,
            'p90_error': sub[ERR].quantile(0.90) if len(sub) else None,
            'median_bias': sub['test_bias'].median() if len(sub) else None,
            'mean_bias': sub['test_bias'].mean() if len(sub) else None,
        })
    out = pd.DataFrame(rows)
    print(f'\n-- {label} --')
    print(out.to_string(index=False))
    return out

threshold_compare(W, 'OVERALL')
for patch in [f'{a}->{b}' for a, b in PATCHES]:
    threshold_compare(W.loc[W['patch'] == patch], f'patch {patch}')

# ── 6) Patch-specific summary ──
print('\n=== Patch-specific summary ===')
patch_order = [f'{a}->{b}' for a, b in PATCHES]
patch_rows = []
for patch in patch_order:
    sub = W.loc[W['patch'] == patch].dropna(subset=['cv_after'])
    if sub.empty:
        continue
    le = sub.loc[sub['cv_after'] <= 0.15, ERR]
    gt = sub.loc[sub['cv_after'] > 0.15, ERR]
    d = sub.dropna(subset=[ERR])
    patch_rows.append({
        'patch': patch,
        'median_cv': sub['cv_after'].median(),
        'p75_cv': sub['cv_after'].quantile(0.75),
        'p90_cv': sub['cv_after'].quantile(0.90),
        'median_oos_error': d[ERR].median() if len(d) else None,
        'p75_oos_error': d[ERR].quantile(0.75) if len(d) else None,
        'p90_oos_error': d[ERR].quantile(0.90) if len(d) else None,
        'corr_cv_error_pearson': d['cv_after'].corr(d[ERR], method='pearson') if len(d) >= 5 else None,
        'corr_cv_error_spearman': d['cv_after'].corr(d[ERR], method='spearman') if len(d) >= 5 else None,
        'pct_windows_cv_gt_15': float((sub['cv_after'] > 0.15).mean()),
        'median_error_cv_le_15': le.median() if len(le) else None,
        'median_error_cv_gt_15': gt.median() if len(gt) else None,
        'n_backtests': len(sub),
    })
patch_summary = pd.DataFrame(patch_rows)
print(patch_summary.to_string(index=False))

# ── 7) CV buckets ──
print('\n=== Error by CV bucket ===')
bins = [-np.inf, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, np.inf]
labels = ['<5%', '5-10%', '10-15%', '15-20%', '20-25%', '25-30%', '>30%']
Wb = W.dropna(subset=['cv_after', ERR]).copy()
Wb['cv_bucket'] = pd.cut(Wb['cv_after'], bins=bins, labels=labels)

def bucket_table(df, label):
    print(f'\n-- {label} --')
    if df.empty:
        print('(empty)')
        return
    g = df.groupby('cv_bucket', observed=True).agg(
        n=('cv_after', 'count'),
        median_error=(ERR, 'median'),
        p75_error=(ERR, lambda s: s.quantile(0.75)),
        p90_error=(ERR, lambda s: s.quantile(0.90)),
        median_bias=('test_bias', 'median'),
    )
    print(g.to_string())

bucket_table(Wb, 'OVERALL')
for patch in ['1->7', '7->14', '14->30', '30->60', '60->90', '90->120']:
    bucket_table(Wb.loc[Wb['patch'] == patch], f'patch {patch}')

# ── 8) Special focus 1->7 ──
print('\n=== 1->7 deep dive ===')
w17 = wf_df.loc[(wf_df['patch'] == '1->7') & (wf_df['n_test_cohorts'] > 0)].copy()
# pivot horizons
piv = (
    w17.pivot_table(
        index=['brand', 'population', 'cutoff_as_of', 'goal', 'cv_after'],
        columns='test_horizon',
        values=ERR,
        aggfunc='first',
    )
    .reset_index()
)
piv = piv.rename(columns={7: 'next_7_error', 14: 'next_14_error', 30: 'next_30_error'})
print(piv.sort_values(['brand', 'population', 'cutoff_as_of']).to_string(index=False))

def q_band(df, lo, hi, name):
    sub = df.loc[(df['cv_after'] > lo) & (df['cv_after'] <= hi) & (df['test_horizon'] == PRIMARY_H)]
    print(f'\nQuestion {name}: CV in ({lo:.0%}, {hi:.0%}]  n={len(sub)}')
    if sub.empty:
        print('  insufficient data')
        return
    print(f"  median {ERR}={sub[ERR].median():.4f}  p75={sub[ERR].quantile(0.75):.4f}  "
          f"p90={sub[ERR].quantile(0.90):.4f}  mean={sub[ERR].mean():.4f}")

w17p = w17.loc[w17['test_horizon'] == PRIMARY_H]
q_band(w17p, 0.15, 0.20, 'A 15-20%')
q_band(w17p, 0.20, 0.25, 'B 20-25%')
q_band(w17p, 0.25, 1.00, 'C >25%')
print('\nQuestion D: look at bucket table for 1->7 — only call a cliff if error jumps materially.')

# ── 9) Population / sample size ──
print('\n=== By brand × population (primary horizon) ===')
print(
    W.groupby(['brand', 'population'], observed=True)
     .agg(n=('cv_after', 'count'),
          median_cv=('cv_after', 'median'),
          median_err=(ERR, 'median'),
          pearson=('cv_after', lambda s: s.corr(W.loc[s.index, ERR], method='pearson') if len(s)>=5 else np.nan))
     .to_string()
)

print('\n=== Similar CV, large vs small N (exploratory) ===')
W2 = W.dropna(subset=['cv_after', ERR, 'n_training_users']).copy()
W2['cv_band'] = pd.cut(W2['cv_after'], bins=[-np.inf, 0.10, 0.15, 0.20, 0.30, np.inf],
                       labels=['<=10%', '10-15%', '15-20%', '20-30%', '>30%'])
W2['n_band'] = pd.cut(W2['n_training_users'], bins=[-np.inf, 2000, 10000, np.inf],
                      labels=['small<=2k', 'mid', 'large>10k'])
print(
    W2.groupby(['cv_band', 'n_band'], observed=True)
      .agg(n=(ERR, 'count'), median_err=(ERR, 'median'))
      .to_string()
)
print('RealPrize Web:')
print(
    W2.loc[(W2['brand'] == 'realprize') & (W2['population'] == 'Web'),
           ['patch', 'cv_after', 'n_training_users', ERR, 'flagged_using_existing_logic']]
    .sort_values(ERR, ascending=False)
    .head(30)
    .to_string(index=False)
)

# ── 10) Goal stability ──
print('\n=== Goal stability across cutoffs ===')
stab_rows = []
for (brand, pop, patch), g in W.groupby(['brand', 'population', 'patch']):
    goals = g['goal'].dropna()
    if len(goals) < 3:
        continue
    stab_rows.append({
        'brand': brand, 'population': pop, 'patch': patch,
        'n_windows': len(goals),
        'goal_median': goals.median(),
        'goal_cv': float(goals.std(ddof=0) / goals.mean()) if goals.mean() else None,
        'goal_iqr': float(goals.quantile(0.75) - goals.quantile(0.25)),
        'goal_range': float(goals.max() - goals.min()),
        'median_cv_after': g['cv_after'].median(),
    })
stability_df = pd.DataFrame(stab_rows)
print(stability_df.sort_values('goal_cv', ascending=False).to_string(index=False))

# ── 11) Plots ──
print('\n=== Plots ===')
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(W['cv_after'], W[ERR], alpha=0.55, s=25)
ax.axvline(0.15, color='red', ls='--', label='CV=0.15')
ax.set_xlabel('cv_after (in-sample)')
ax.set_ylabel(f'{ERR} (OOS, H={PRIMARY_H})')
ax.set_title('CV after → out-of-sample error')
ax.legend()
plt.tight_layout()
plt.show()

focus_patches = ['1->7', '7->14', '14->30', '30->60', '60->90', '90->120']
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=False, sharey=False)
for i, patch in enumerate(focus_patches):
    ax = axes[i // 3][i % 3]
    sub = W.loc[W['patch'] == patch]
    ax.scatter(sub['cv_after'], sub[ERR], alpha=0.6, s=20)
    ax.axvline(0.15, color='red', ls='--', lw=1)
    ax.set_title(patch)
    ax.set_xlabel('cv_after')
    ax.set_ylabel(ERR)
plt.suptitle('CV vs OOS error by patch')
plt.tight_layout()
plt.show()

# Error by CV bucket boxplot
fig, ax = plt.subplots(figsize=(9, 5))
order = [l for l in labels if l in set(Wb['cv_bucket'].astype(str))]
data = [Wb.loc[Wb['cv_bucket'] == l, ERR].dropna().values for l in labels]
ax.boxplot(data, labels=labels, showfliers=False)
ax.set_title(f'OOS {ERR} by CV bucket (primary H={PRIMARY_H})')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Rolling goal stability examples: focus cases
FOCUS_PLOT = [
    ('realprize', 'Web', '1->7'),
    ('realprize', 'Web', '14->30'),
    ('lonestar', 'Web', '1->7'),
    ('lonestar', 'Blended', '1->7'),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, (brand, pop, patch) in enumerate(FOCUS_PLOT):
    ax = axes[i // 2][i % 2]
    sub = W.loc[(W['brand'] == brand) & (W['population'] == pop) & (W['patch'] == patch)].copy()
    if sub.empty:
        ax.set_title(f'{brand} {pop} {patch} (no data)')
        continue
    sub['cutoff_dt'] = pd.to_datetime(sub['cutoff_as_of'])
    sub = sub.sort_values('cutoff_dt')
    ax.plot(sub['cutoff_dt'], sub['goal'], marker='o', label='goal (mean_after)')
    ax2 = ax.twinx()
    ax2.plot(sub['cutoff_dt'], sub['cv_after'], color='orange', alpha=0.7, label='cv_after')
    ax.set_title(f'{brand} | {pop} | {patch}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('goal')
    ax2.set_ylabel('cv_after')
plt.suptitle('Rolling goal stability (left axis) vs CV (right)')
plt.tight_layout()
plt.show()

# 1->7 deep dive scatter
fig, ax = plt.subplots(figsize=(7, 5))
sub = W.loc[W['patch'] == '1->7']
ax.scatter(sub['cv_after'], sub[ERR], alpha=0.65)
ax.axvline(0.15, color='red', ls='--')
ax.axvline(0.20, color='grey', ls=':')
ax.axvline(0.25, color='grey', ls=':')
ax.set_title('1->7: CV after vs OOS error')
ax.set_xlabel('cv_after')
ax.set_ylabel(ERR)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# FINAL DECISION TABLE + answers (descriptive only)
# ══════════════════════════════════════════════════════════════

def _interpret(row):
    n = row['n_backtests']
    if n is None or n < 8:
        return 'Insufficient historical observations'
    sp = row['corr_cv_error_spearman']
    pe = row['corr_cv_error_pearson']
    corr = sp if sp is not None and np.isfinite(sp) else pe
    le = row['median_error_cv_le_15']
    gt = row['median_error_cv_gt_15']
    notes = []
    if corr is None or not np.isfinite(corr):
        notes.append('Weak/unclear CV–error relationship')
    elif corr >= 0.45:
        notes.append('Higher CV strongly associated with worse future prediction')
    elif corr >= 0.25:
        notes.append('Moderate relationship')
    elif corr >= 0.10:
        notes.append('Weak relationship')
    else:
        notes.append('No meaningful relationship detected')
    if le is not None and gt is not None and np.isfinite(le) and np.isfinite(gt):
        if gt > le * 1.25:
            notes.append('CV>15% group has materially higher median OOS error')
        elif gt <= le * 1.10:
            notes.append('CV >15% does not materially reduce predictive accuracy')
    return ' | '.join(notes)

final = patch_summary.copy()
final['interpretation'] = final.apply(_interpret, axis=1)
final_cols = [
    'patch', 'median_cv', 'p90_cv', 'median_oos_error', 'p90_oos_error',
    'corr_cv_error_pearson', 'corr_cv_error_spearman',
    'median_error_cv_le_15', 'median_error_cv_gt_15', 'n_backtests', 'interpretation',
]
print('=== FINAL summary by patch (primary OOS horizon) ===')
print(final[final_cols].to_string(index=False))

# Cases: high CV but good OOS / low CV but bad OOS
print('\n=== High CV but relatively low OOS error (top examples) ===')
hi = W.loc[W['cv_after'] > 0.15].copy()
if not hi.empty:
    hi['err_rank'] = hi[ERR].rank(pct=True)
    print(hi.nsmallest(15, ERR)[
        ['brand', 'population', 'patch', 'cutoff_as_of', 'cv_after', 'goal', ERR, 'test_bias']
    ].to_string(index=False))

print('\n=== Low CV but relatively high OOS error (top examples) ===')
lo = W.loc[W['cv_after'] <= 0.10].copy()
if not lo.empty:
    print(lo.nlargest(15, ERR)[
        ['brand', 'population', 'patch', 'cutoff_as_of', 'cv_after', 'goal', ERR, 'test_bias']
    ].to_string(index=False))

print('\n=== Concise answers (evidence from this backtest only) ===')
# 1
d = W.dropna(subset=['cv_after', ERR])
sp = d['cv_after'].corr(d[ERR], method='spearman') if len(d) >= 5 else np.nan
print(f'1) Does CV predict future goal error?  Overall Spearman({ERR})={sp:.3f} on n={len(d)}. '
      f'See per-patch correlations in the table.')
# 2
le = d.loc[d['cv_after'] <= 0.15, ERR].median()
gt = d.loc[d['cv_after'] > 0.15, ERR].median()
print(f'2) Does 15% threshold separate reliability?  median {ERR}: CV<=15% → {le:.4f} ; CV>15% → {gt:.4f}')
# 3
print('3) Relationship by patch: see FINAL table (corr + interpretation). Early vs mature often differ.')
# 4
s17 = final.loc[final['patch'] == '1->7']
if not s17.empty:
    r = s17.iloc[0]
    print(f"4) 1->7: median_cv={r['median_cv']:.3f}, corr_spearman={r['corr_cv_error_spearman']}, "
          f"interp={r['interpretation']}")
else:
    print('4) 1->7: insufficient rows')
# 5-6 already printed examples
print('5) High-CV but accurate cases: see table above (if any).')
print('6) Low-CV but inaccurate cases: see table above (if any).')
# 7
early = final.loc[final['patch'].isin(['1->7', '7->14', '14->30'])]
mature = final.loc[final['patch'].isin(['90->120', '120->150', '150->180', '180->270', '270->365'])]
print(
    '7) Patch-specific CV expectations?  Compare early vs mature median_cv / corr in FINAL table. '
    'Only treat as evidence for eventually considering patch-specific expectations if early patches '
    'show high natural CV without matching OOS-error penalty. Do NOT change production from this notebook.'
)
print('\nReminder: this experiment does not change Marketing Goals methodology.')


In [ ]:
# ══════════════════════════════════════════════════════════════
# EXPORT — backtest CSVs (distinct experiment_tag; no overwrites)
# ══════════════════════════════════════════════════════════════

from datetime import datetime
from pathlib import Path
import re

_BRAND_SLUG = {'realprize': 'rp', 'lonestar': 'ls'}
brand_slug = '_'.join(
    _BRAND_SLUG.get(b, re.sub(r'[^a-z0-9]+', '', b.lower())[:6])
    for b in RUN_BRANDS
)
exported_at = datetime.now()
run_ts = exported_at.strftime('%H%M%S')
run_tag = f"{AS_OF_DATE.date()}_{brand_slug}_{EXPERIMENT_TAG}_{run_ts}"

files = {
    'windows': f'cv_oos_backtest_windows_{run_tag}.csv',
    'points': f'cv_oos_backtest_points_{run_tag}.csv',
    'patch_summary': f'cv_oos_backtest_patch_summary_{run_tag}.csv',
    'final': f'cv_oos_backtest_final_{run_tag}.csv',
}
wf_df.to_csv(files['windows'], index=False)
pt_df.to_csv(files['points'], index=False)
patch_summary.to_csv(files['patch_summary'], index=False)
final[final_cols].to_csv(files['final'], index=False)

print('Run tag:', run_tag)
for p in files.values():
    print(' ', p)

run_candidates = [
    Path('/Users/leejerusalmy/Library/CloudStorage/'
         'GoogleDrive-lee@realplayltd.com/My Drive/lee_project/'
         'marketing_goals/runs'),
    Path('/content/drive/MyDrive/lee_project/marketing_goals/runs'),
    Path('/content/drive/My Drive/lee_project/marketing_goals/runs'),
]
drive_runs = next((c for c in run_candidates if c.is_dir()), None)
if drive_runs is not None:
    out_dir = drive_runs / run_tag
    out_dir.mkdir(parents=True, exist_ok=True)
    wf_df.to_csv(out_dir / 'cv_oos_backtest_windows.csv', index=False)
    pt_df.to_csv(out_dir / 'cv_oos_backtest_points.csv', index=False)
    patch_summary.to_csv(out_dir / 'cv_oos_backtest_patch_summary.csv', index=False)
    final[final_cols].to_csv(out_dir / 'cv_oos_backtest_final.csv', index=False)
    pd.DataFrame([{
        'run_tag': run_tag,
        'as_of_date': str(AS_OF_DATE.date()),
        'brands': ','.join(RUN_BRANDS),
        'experiment_tag': EXPERIMENT_TAG,
        'primary_test_horizon': PRIMARY_H,
        'cutoff_freq_days': CUTOFF_FREQ_DAYS,
        'cutoff_span_days': CUTOFF_SPAN_DAYS,
        'n_cutoffs': len(cutoff_dates),
        'exported_at': exported_at.isoformat(timespec='seconds'),
        'purpose': 'cv_predicts_oos_goal_error_walkforward',
        'goal_definition': 'patch mean_after (weighted mean growth_ratio after CV cleanup)',
    }]).to_csv(out_dir / 'run_meta.csv', index=False)
    print('Saved Drive folder:', out_dir)
else:
    print('Drive runs/ not mounted — working-dir CSVs only.')
